# SSD demo: sparse-spike deconvolution

Noiseless sparse-spike deconvolution demo using `seis.mat`. This is a compact Python version for public reproducibility, not the full MATLAB experiment script.

In [ ]:
EPOCHS = 10
PROFILE_TRACE = 100
TRACE_START = 0
TRACE_COUNT = 30
TIME_START = 200
TIME_STOP = 800
DT = 0.001
T0 = TIME_START * DT


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from demo_utils import (
    DATA_DIR,
    FIGURE_DIR,
    RESULT_DIR,
    extract_positions,
    load_mat_array,
    pick_interval_section,
    plot_position_points,
    save_mat,
    sparse_spike_deconvolution,
    to_interval_trace_time,
    train_position_demo,
    train_reflectivity_demo,
    wigb,
)

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 14,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.linewidth": 0.9,
    "savefig.dpi": 300,
})
FIGURE_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)


## Load noiseless data

In [ ]:
ref = to_interval_trace_time(load_mat_array(DATA_DIR / "ref.mat", "ref"))
rbp = to_interval_trace_time(load_mat_array(DATA_DIR / "rbp_180_200.mat", "rbp_180_200"))
seis = to_interval_trace_time(load_mat_array(DATA_DIR / "seis.mat", "seis"))

ref_sec = pick_interval_section(ref, PROFILE_TRACE, TRACE_START, TRACE_COUNT, TIME_START, TIME_STOP)
rbp_sec = pick_interval_section(rbp, PROFILE_TRACE, TRACE_START, TRACE_COUNT, TIME_START, TIME_STOP)
seis_sec = pick_interval_section(seis, PROFILE_TRACE, TRACE_START, TRACE_COUNT, TIME_START, TIME_STOP)
print("seis", seis.shape, "ref", ref.shape)

## Run compact SSD

In [ ]:
ssd_sec = sparse_spike_deconvolution(seis_sec, fm=30.0, dt=DT, lam=0.002, n_iter=80)
save_mat(RESULT_DIR / "quick_ssd_noiseless.mat", prediction=ssd_sec.astype(np.float32))
print("ssd_sec", ssd_sec.shape)

## Paper-style SSD figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
wigb(seis_sec, dt=DT, t0=T0, scale=0.75, ax=axes[0], linewidth=0.6, panel_label="(a)")
wigb(rbp_sec, dt=DT, t0=T0, scale=0.75, ax=axes[1], linewidth=0.6, panel_label="(b)")
wigb(ssd_sec, dt=DT, t0=T0, scale=0.75, ax=axes[2], linewidth=0.6, panel_label="(c)")
for ax in axes[1:]:
    ax.set_ylabel("")
fig.savefig(FIGURE_DIR / "ssd_demo.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), constrained_layout=True)
plot_position_points(ref_sec, ax=axes[0], color="black", size=8, panel_label="(a)", dt=DT, t0=T0)
plot_position_points(ssd_sec, ax=axes[1], color="black", size=8, panel_label="(b)", dt=DT, t0=T0)
fig.savefig(FIGURE_DIR / "ssd_positions.png", bbox_inches="tight")
plt.show()